In [1]:
from keyphrase_curation import *
from keyphrase_curation.controller.annotation import *

alex = config["dataset_path"] + "/anotations/alexandre/marijuana_legalization.json"
akira = config["dataset_path"] + "/anotations/akira/marijuana_legalization.json"
print(config["attributions_filepath"])

alex_controller = AnnotationController("marijuana_legalization", "alexandre")
akira_controller = AnnotationController("marijuana_legalization", "akira")

/home/romario/keyphrase_curation/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/romario/keyphrase_curation/dataset/attributions.toml


In [2]:
alex_clustering = [4, 3, 3, 3, 9, 11, 23, 17, 17, 30, 26, 4, 26, 26, 6, 22, 11, 17, 3, 27, 16, 1, 5, 16, 8, 8, 8, 24, 24, 5, 8, 8, 19, 5, 5, 24, 5, 24, 22, 6, 19, 31, 32, 32, 32, 32, 31, 31, 31, 31, 20, 18, 13, 29, 19, 32, 18, 18, 17, 28, 12, 13, 7, 25, 18, 12, 13, 3, 11, 33, 33, 14, 13, 33, 7, 14, 33, 1, 15, 13, 2, 20, 13, 1, 1, 33, 33, 14, 15, 12, 25, 7, 33, 13, 16, 12, 1, 26, 21, 15, 2, 33, 2, 19, 33, 26, 10, 28, 33, 28, 21, 28, 19, 9, 10, 19, 2, 2, 19, 1, 33, 9, 32, 19, 10, 2, 7, 31, 9, 10, 19, 10, 28, 2, 22, 7, 21, 19, 29, 29]
akira_clustering = [2, 15, 3, 15, 4, 17, 13, 10, 10, 26, 12, 2, 2, 2, 3, 20, 17, 14, 15, 3, 1, 5, 23, 1, 18, 16, 23, 15, 19, 1, 3, 21, 6, 14, 19, 19, 14, 19, 16, 3, 29, 29, 29, 29, 29, 29, 29, 29, 29, 29, 31, 22, 12, 30, 25, 25, 11, 11, 11, 32, 7, 12, 13, 7, 10, 7, 12, 17, 17, 33, 33, 13, 12, 11, 13, 13, 10, 5, 14, 2, 8, 21, 12, 5, 5, 33, 33, 13, 9, 7, 1, 5, 33, 2, 5, 7, 13, 12, 23, 14, 8, 33, 8, 28, 33, 2, 33, 32, 33, 32, 27, 32, 28, 4, 33, 28, 33, 8, 28, 5, 11, 4, 2, 29, 8, 4, 13, 28, 9, 4, 28, 33, 9, 33, 24, 13, 27, 28, 30, 30]

In [3]:
from sklearn.metrics.cluster import fowlkes_mallows_score
from sklearn.metrics.cluster import adjusted_mutual_info_score
from sklearn.metrics.cluster import adjusted_rand_score

def jaccard_dist(ksa, ksb):
    return round(len(ksa.intersection(ksb)) / len(ksa.union(ksb)), 2)


def is_match(ksa, ksb):
    return jaccard_dist(ksa, ksb) > 0.2


def ari(ca, cb):
    return adjusted_rand_score(ca, cb)


def nmi(ca, cb):
    return adjusted_mutual_info_score(ca, cb)


def fms(ca, cb):
    return fowlkes_mallows_score(ca, cb)

print(round(ari(akira_clustering, alex_clustering), 2))
print(round(nmi(akira_clustering, alex_clustering), 2))
print(round(fms(akira_clustering, alex_clustering), 2))

0.39
0.51
0.41


In [4]:
alex_cs = list(alex_controller.get_cluster_selection().values())[:-1]
akira_cs = list(akira_controller.get_cluster_selection().values())[:-1]

In [5]:
len(alex_cs) == len(akira_cs)

True

In [6]:
def make_data(coder_id, labels):
    data = []
    if len(labels) == 0:
        print("Empty keys")
    for id in range(len(labels)):
        data.append((f'Coder_{coder_id}', id, frozenset([labels[id]])))
    return data


V_ALEX = alex_cs
V_AKIRA = akira_cs

# sklearn kappa
from sklearn.metrics import cohen_kappa_score
print("Kappa: V_A-V_V", cohen_kappa_score(V_ALEX, V_AKIRA))

# krippendorff's alpha
from nltk.metrics.agreement import AnnotationTask
from nltk.metrics.distance import jaccard_distance

va_data = make_data(1, V_ALEX)
vv_data = make_data(2, V_AKIRA)

task = AnnotationTask(data=va_data + vv_data, distance=jaccard_distance)
print("Alpha Jaccard V_A-V_V", round(task.alpha(),2))

Kappa: V_A-V_V 0.25
Alpha Jaccard V_A-V_V 0.26


In [7]:
akira_ks = [0] * len(akira_controller.get_keyphrases_selection())
alex_ks = [0] * len(alex_controller.get_keyphrases_selection())

akira_ks_data = akira_controller.get_keyphrases_selection()
alex_ks_data = alex_controller.get_keyphrases_selection()

for k in akira_ks_data:
    akira_ks[int(k) - 1] = (akira_ks_data[k]["selected1"], akira_ks_data[k]["selected2"])
for k in alex_ks_data:
    alex_ks[int(k) - 1] = (alex_ks_data[k]["selected1"], alex_ks_data[k]["selected2"])


In [8]:
len(akira_ks) == len(alex_ks)

True

In [9]:
for i in range(len(akira_ks)):
    print(akira_ks[i], alex_ks[i])

(10, 0) (10, 0)
(54, 0) (54, 0)
(90, 0) (61, 64)
(133, 0) (60, 0)
(6, 0) (6, 0)
(137, 0) (99, 0)
(120, 78) (120, 0)
(131, 0) (131, 0)
(29, 0) (36, 0)
(80, 0) (11, 0)
(103, 0) (103, 0)
(51, 0) (51, 0)
(126, 0) (5, 0)
(2, 3) (2, 0)
(41, 0) (41, 0)
(8, 0) (9, 0)
(21, 0) (21, 0)
(79, 0) (34, 37)
(55, 0) (55, 0)
(135, 0) (16, 0)
(75, 0) (136, 0)
(25, 0) (25, 0)
(23, 0) (23, 0)
(76, 0) (88, 0)
(1, 0) (106, 0)
(15, 0) (15, 20)
(26, 0) (26, 0)
(33, 0) (33, 0)
(123, 0) (123, 0)
(7, 0) (7, 0)
(52, 0) (52, 0)
(65, 59) (57, 59)
(0, 0) (0, 0)


In [10]:
from nltk.metrics.distance import jaccard_distance
from nltk.metrics.agreement import AnnotationTask


def make_data(coder_id, labels):
    data = []
    if len(labels) == 0:
        print("Empty keys")
    for id in range(len(labels)):
        data.append((f'Coder_{coder_id}', id, frozenset(labels[id])))
    return data


akira_selections = akira_ks
alex_selections = alex_ks

alex_data = make_data(1, alex_selections)
akira_data = make_data(2, akira_selections)

# krippendorff's alpha

task = AnnotationTask(data=alex_data + akira_data, distance=jaccard_distance)
print("Kappa Fleiss Alex-Akira", round(task.multi_kappa(),2))

Kappa Fleiss Alex-Akira 0.65
